# Research Assistant — MCP + Gemini

End-to-end agentic app: Gemini decides which tools to call across three MCP servers (filesystem, fetch, custom research tools). The LLM drives the tool selection — no hardcoded flow.

**Theme:** Research Assistant — read/write local files, fetch web content, analyze text.

Run cells top to bottom.

## 1. Install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio"

## 2. Set GOOGLE_API_KEY

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key set:", bool(os.environ.get("GOOGLE_API_KEY")))

## 3. Confirm Node / NPM availability

In [ ]:
!node --version
!npx --version

In [ ]:
# uncomment if node/npx not found above
# !apt-get -qq update && apt-get -qq install -y nodejs npm
# !node --version && npx --version

## 4. Working directory + sample files

In [ ]:
import asyncio
import nest_asyncio
from pathlib import Path

nest_asyncio.apply()

WORKDIR = "/content/research"
Path(WORKDIR).mkdir(exist_ok=True)

(Path(WORKDIR) / "notes.txt").write_text(
    "LLMs are trained on large text corpora using self-supervised learning.\n"
    "The transformer architecture uses attention to model long-range dependencies.\n"
    "RLHF is a technique used to align language models with human preferences.\n"
    "Retrieval-augmented generation grounds model outputs in external documents.\n"
    "Agents use tool-calling to interact with external systems and APIs.\n"
)

print("workdir:", WORKDIR)
print("files:", [f.name for f in Path(WORKDIR).iterdir()])

## 5. Custom MCP server (research_tools)

Three tools: `word_count`, `format_citation`, `extract_keywords`.

In [ ]:
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import List, Dict
    import re
    from collections import Counter

    mcp = FastMCP(name="research_tools")

    @mcp.tool
    def word_count(text: str) -> Dict[str, int]:
        \"\"\"Count words, lines, and characters in a block of text.\"\"\"
        return {
            "words": len(text.split()),
            "lines": len(text.splitlines()),
            "chars": len(text),
        }

    @mcp.tool
    def format_citation(title: str, url: str, author: str = "Unknown", year: str = "2024") -> str:
        \"\"\"Format an APA-style citation string.\"\"\"
        return f"{author} ({year}). {title}. Retrieved from {url}"

    @mcp.tool
    def extract_keywords(text: str, top_n: int = 5) -> List[str]:
        \"\"\"Return the top_n most frequent non-stopword words from text.\"\"\"
        stopwords = {
            "the", "a", "an", "is", "in", "of", "and", "to", "for", "with",
            "on", "at", "by", "this", "that", "are", "was", "it", "as", "from",
            "be", "have", "has", "been", "or", "not", "but", "its", "they", "which"
        }
        words = re.findall(r"[a-z]+", text.lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 3]
        return [w for w, _ in Counter(filtered).most_common(top_n)]

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print("Wrote:", server_path)

## 6. Connect to MCP servers

Two third-party servers + custom:
- `@modelcontextprotocol/server-filesystem` — read/write files in WORKDIR
- `@modelcontextprotocol/server-fetch` — fetch URLs and convert to markdown
- `custom_mcp_server.py` — research_tools (word_count, format_citation, extract_keywords)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "fetch": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-fetch"],
    },
}

print("connections defined")

## 7. Add custom server + inspect all tools

In [ ]:
mcp_connections["research_tools"] = {
    "transport": "stdio",
    "command": "python",
    "args": [str(server_path)],
}

client2 = MultiServerMCPClient(mcp_connections)
tools2 = asyncio.get_event_loop().run_until_complete(client2.get_tools())

print("Tool count:", len(tools2))
print([t.name for t in tools2])

## 8. Build the Gemini agent

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
)

async def run_agent(query: str) -> str:
    async with MultiServerMCPClient(mcp_connections) as client:
        tools = await client.get_tools()
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": [("user", query)]})
        return result["messages"][-1].content

def ask(query: str) -> str:
    return asyncio.get_event_loop().run_until_complete(run_agent(query))

print("agent ready")

## 9. Run the research assistant

Each query below shows the LLM picking different combinations of tools.

### Query 1 — filesystem: list files

In [ ]:
result = ask("List all files in the research directory.")
print(result)

### Query 2 — filesystem + research_tools: read, analyze, write

In [ ]:
result = ask(
    "Read notes.txt, extract the top 5 keywords using extract_keywords, "
    "then write a new file called keywords.txt containing those keywords one per line."
)
print(result)

### Query 3 — filesystem + research_tools: word count + citation

In [ ]:
result = ask(
    "Read notes.txt and count its words using word_count. "
    "Then format a citation for it: title='LLM Research Notes', "
    "url='local://notes.txt', author='Research Team', year='2025'."
)
print(result)

### Query 4 — fetch + research_tools: fetch web page and extract keywords

In [ ]:
result = ask(
    "Fetch https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture) "
    "and extract the top 8 keywords from the content you retrieve. "
    "Then save them to transformer_keywords.txt."
)
print(result)

## 10. Multi-step demo — full research pipeline

This shows the agent chaining all three servers in a single request.

In [ ]:
result = ask(
    "Do these steps in order:\n"
    "1. Read notes.txt\n"
    "2. Extract the top 5 keywords from it\n"
    "3. Count the words in it\n"
    "4. Fetch https://en.wikipedia.org/wiki/Large_language_model and extract 5 more keywords from the intro\n"
    "5. Format a citation: title='LLM Notes', url='local://notes.txt', author='Research Team', year='2025'\n"
    "6. Write a file report.txt that includes: the keywords from notes, keywords from Wikipedia, the word count, and the citation"
)
print(result)

In [ ]:
report = Path(WORKDIR + "/report.txt")
if report.exists():
    print(report.read_text())
else:
    print("report.txt not found — check agent output above")